In [ ]:
from datetime import datetime, UTC
import os.path

from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# If modifying these scopes, delete the file token.json.
SCOPES = ["https://www.googleapis.com/auth/calendar.readonly"]
TOKEN_SAVE_PATH = "gcal_token.json"


def main():
    """Shows basic usage of the Google Calendar API.
    Prints the start and name of the next 10 events on the user's calendar.
    """
    creds = None
    # The file token.json stores the user's access and refresh tokens, and is
    # created automatically when the authorization flow completes for the first
    # time.
    if os.path.exists(TOKEN_SAVE_PATH):
        creds = Credentials.from_authorized_user_file(TOKEN_SAVE_PATH, SCOPES)
    # If there are no (valid) credentials available, let the user log in.
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                "../credentials.json", SCOPES
            )
            creds = flow.run_local_server(port=0)
        # Save the credentials for the next run
        with open(TOKEN_SAVE_PATH, "w") as token:
            token.write(creds.to_json())

    try:
        service = build("calendar", "v3", credentials=creds)

        # Call the Calendar API
        now = datetime.now(tz=UTC).isoformat()
        print("Getting the upcoming 10 events")
        events_result = (
            service.events()
            .list(
                calendarId="primary",
                timeMin=now,
                maxResults=10,
                singleEvents=True,
                orderBy="startTime",
            )
            .execute()
        )
        events = events_result.get("items", [])

        if not events:
            print("No upcoming events found.")
            return

        # Prints the start and name of the next 10 events
        for event in events:
            start = event["start"].get("dateTime", event["start"].get("date"))
            print(start, event["summary"])

    except HttpError as error:
        print(f"An error occurred: {error}")


if __name__ == "__main__":
    main()

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=169585957198-oi8ve57ggmdoct8omqr5jil37srsfqn1.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A54813%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcalendar.readonly&state=0u6gtZgFjFI0cwD12zkeWZPcEeu58U&code_challenge=piG6Z8-0q81WtI3poTupTqFPImHmYrH78KAXNBhdBH8&code_challenge_method=S256&access_type=offline
Getting the upcoming 10 events
2026-09-17T10:00:00+01:00 Office
2026-09-18T10:00:00+01:00 David Lloyd
2026-09-18T18:00:00+01:00 emma
2026-09-19T13:00:00+01:00 gym
2026-09-19T20:00:00+01:00 VG drinks
2026-09-20T10:30:00+01:00 Net Social
2026-09-21T10:00:00+01:00 David Lloyd
2026-09-21T13:00:00+01:00 gym
2026-09-22 jemma's half birthday
2026-09-22T10:00:00+01:00 Office
